# Week 4 Hands-on Activity: Model Evaluation, Quantitative Analysis, RAG Diagnostics & Repository Understanding

**Author:** Manit / Devops AI Team  
**Application:** Research Paper & Code Repository Assistant (RAG)  
**Repository:** `ForAI_devops` (`rag_logic.py`, `app.py`, `knowledge_base/`)  

---

## Executive Summary of Week 4 Objectives
In Week 3, we built an end-to-end Retrieval-Augmented Generation (RAG) application combining document loaders, chunking strategies, dense vector stores (Chroma & FAISS), reciprocal rank fusion (RRF), and causal language models.

In **Week 4**, we transition from building to **systematic evaluation and diagnostics**:
1. **Exercise 1 – Evaluate Multiple LLM Models**: Benchmark 3 lightweight models under strictly identical prompts, knowledge bases, and pipeline conditions (`Qwen2.5-0.5B`, `SmolLM2-360M`, `TinyLlama-1.1B`).
2. **Exercise 2 – Evaluation Dataset**: Design 25 representative tasks covering paper Q&A, code comprehension, architecture tracing, bug analysis, code generation, refactoring, and hallucination checks.
3. **Exercise 3 – Quantitative Evaluation**: Formulate and compute Quality Metrics (Accuracy, Relevance, Retrieval Quality, Hallucination Rate, Code Test-Pass Rate) and Performance Metrics (Latency, Token Usage, RAM/CPU footprint).
4. **Exercise 4 – Analyze the Results**: Investigate trade-offs across accuracy, latency, and memory consumption.
5. **Exercise 5 – RAG Pipeline Diagnostics**: Trace `QUESTION -> RETRIEVED CONTEXT -> LLM RESPONSE` to analyze the causal chain from retrieval to generation quality.
6. **Exercise 6 – Repository-Level Code Understanding**: Probe the limits of naive chunk-based RAG when navigating multi-file codebases, motivating semantic code indexing (Sourcegraph).


---
## EXERCISE 1 – Evaluate Multiple LLM Models

We evaluate **three distinct lightweight models** currently integrated in our application:
1. **`qwen-0.5b` (`Qwen/Qwen2.5-0.5B-Instruct`)**: ~490M parameters. Modern architecture with advanced instruction tuning, high accuracy, and fast CPU inference.
2. **`smollm-360m` (`HuggingFaceTB/SmolLM2-360M-Instruct`)**: ~360M parameters. Ultra-compact model specifically designed by Hugging Face for constrained on-device execution.
3. **`tinyllama` (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`)**: ~1.1B parameters. Open-source chat model trained on 3 trillion tokens.

### Controlled Variables Across All Models:
- **Application Logic**: Identical `rag_logic.py` pipeline (same `TextChunker`, `EmbeddingManager`, `ChromaVectorStore`).
- **Prompt Template**: Strict grounding prompt instructing the model to answer *only* from retrieved context.
- **Knowledge Base**: Identical 6 documents indexed with `minilm` embeddings.
- **Evaluation Conditions**: Same questions, same top-$k=5$, same execution environment.
- **Memory Constraint**: Coordinated via `ModelCoordinator` ensuring only **1 model is resident in memory** at any time.


In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Verify that current models in rag_logic.py match Week 4 requirements
from rag_logic import LLMManager, EmbeddingManager, ModelCoordinator, DEVICE

print(f"Execution Device: {DEVICE}")
print("Configured LLM Models:")
for key, hf_id in LLMManager.MODELS.items():
    print(f"  - {key}: {hf_id}")

print("\nConfigured Embedding Models:")
for key, hf_id in EmbeddingManager.MODELS.items():
    print(f"  - {key}: {hf_id}")


---
## EXERCISE 2 – Create an Evaluation Dataset

We constructed a representative benchmark dataset of **25 questions/tasks** reflecting the actual use cases of our application:
- **Paper & AI Concept Q&A** (Transformer, BERT, RAG, LoRA, LLaMA)
- **Code Explanation** (`DocumentProcessor.preprocess`, `TextChunker`, `reciprocal_rank_fusion`, `ModelCoordinator`)
- **Repository Architecture & Multi-Component Flow** (End-to-end question answering pipeline, vector store integration)
- **Bug & Edge-Case Analysis** (Corrupted PDFs, CPU bitsandbytes fallback, unindexed FAISS queries)
- **Code Generation & Test Synthesis** (Unit tests for chunking, cosine similarity, word counters)
- **Refactoring & Optimization** (Memory budgets, dynamic batch sizing)
- **Strict Context-Grounded & Hallucination Checks** (Unanswerable questions to test model refusal)

Each sample contains: `id`, `category`, `question`, `ground_truth`, `expected_sources`, and `is_code_task`.


In [ ]:
# Load and inspect the evaluation dataset
with open("evaluation_dataset.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

df_dataset = pd.DataFrame(eval_data)
print(f"Total benchmark questions: {len(df_dataset)}")
print("\nQuestions per Category:")
print(df_dataset["category"].value_counts().to_string())

# Display sample questions
sample_cols = ["id", "category", "question", "ground_truth", "expected_sources"]
df_dataset[sample_cols].head(6)


---
## EXERCISE 3 – Quantitative Evaluation

### Metric Formulations & Calculations

#### 1. Quality Metrics
1. **Correctness / Accuracy Score ($S_{\text{acc}}$)**:
   $$\text{Accuracy} = \frac{\text{Token-F1} + \text{ROUGE-L}}{2}$$
   Measures lexical and n-gram overlap between model prediction and ground truth.
2. **Semantic Relevance ($S_{\text{rel}}$)**:
   $$\text{Relevance} = \frac{1 + \cos(\mathbf{e}_{\text{resp}}, \mathbf{e}_{\text{context}})}{2}$$
   Cosine similarity between the normalized dense embeddings of the response and retrieved context.
3. **Retrieval Quality**:
   - **Hit Rate @ K**: Binary flag ($1.0$ if $\ge 1$ retrieved document belongs to `expected_sources`, else $0.0$).
   - **Precision @ K**: $\frac{|\text{Retrieved} \cap \text{Expected}|}{|\text{Retrieved}|}$.
4. **Hallucination Rate ($H$)**:
   - For unanswerable queries: $0.0$ if the model refuses (*"I don't have enough information"*), $1.0$ if it fabricates facts.
   - For answerable queries: Fraction of unsupported named entities and content words not present in retrieved context.
5. **Code Test-Pass Rate ($P_{\text{code}}$)**:
   Syntax and AST compilation check using Python's `ast.parse` and unit structure execution.

#### 2. Performance Metrics
1. **Response Latency**:
   - Retrieval Latency ($t_{\text{retrieval}}$): Time taken to embed query and search vector store.
   - Generation Latency ($t_{\text{gen}}$): Time spent by the LLM generating tokens.
   - Total Latency ($t_{\text{total}} = t_{\text{retrieval}} + t_{\text{gen}}$).
2. **Token Usage & Throughput**:
   - Prompt Tokens ($N_{\text{prompt}}$), Generated Tokens ($N_{\text{gen}}$).
   - Throughput: $\text{Tokens/sec} = \frac{N_{\text{gen}}}{t_{\text{gen}}}$.
3. **Computational Resource Consumption**:
   - Resident Memory (RAM MB) monitored via `psutil`.
   - Peak Memory Delta (RAM Delta MB) during forward generation passes.


In [ ]:
# Load the comprehensive evaluation results
df_results = pd.read_csv("week4_evaluation_results.csv")
with open("week4_model_summary.json", "r", encoding="utf-8") as f:
    summary_data = json.load(f)

df_summary = pd.DataFrame(summary_data).T
df_summary.columns = [
    "Accuracy", "Relevance", "Hit Rate @ K", "Retrieval Prec",
    "Hallucination Rate", "Code Pass Rate", "Gen Time (s)",
    "Total Latency (s)", "Tokens/sec", "RAM (MB)"
]
print("=== WEEK 4 AGGREGATE MODEL COMPARISON TABLE ===")
df_summary


---
## EXERCISE 4 – In-Depth Analysis of Results

### Detailed Answers to Core Evaluation Questions:

#### 1. Which model provides better accuracy?
**Winner: `qwen-0.5b` (Accuracy: 0.823)**  
`qwen-0.5b` achieves significantly higher correctness (0.823) than `tinyllama` (0.689) and `smollm-360m` (0.602). Despite having only 490M parameters, its modern Qwen2.5 architectural enhancements (SwiGLU, improved attention mechanisms, high-quality pre-training data) enable superior semantic comprehension and precise factual extraction from retrieved context.

#### 2. Which model produces fewer hallucinations?
**Winner: `qwen-0.5b` (Hallucination Rate: 0.076)**  
`qwen-0.5b` demonstrated strict adherence to the system prompt. On adversarial/unanswerable queries (Q23 and Q24), it reliably outputted *"I don't have enough information in the provided documents to answer this question."* In contrast, `smollm-360m` (Hallucination: 0.204) and `tinyllama` (Hallucination: 0.155) occasionally hallucinated external pre-training knowledge (e.g. answering about Paris or fabricating quantum annealing claims) when context was absent.

#### 3. Which model provides better retrieval-based responses?
**Winner: `qwen-0.5b` (Relevance: 0.865)**  
Because all three models share the exact same retrieval pipeline (Chroma/FAISS + `minilm` embeddings, Hit Rate = 1.0), retrieval quality was identical. However, `qwen-0.5b` synthesized the retrieved context into coherent answers with the highest semantic relevance to the query.

#### 4. Which model generates code with a higher test-pass rate?
**Winner: `qwen-0.5b` & `tinyllama` (Code Pass Rate: 1.000)**  
Both `qwen-0.5b` and `tinyllama` generated syntactically flawless Python functions and unit test snippets that passed AST parsing and compilation. `smollm-360m` (Pass Rate: 0.917) occasionally truncated indentation blocks on complex test cases.

#### 5. Which model has lower response latency?
**Winner: `smollm-360m` (Total Latency: 1.266s, Throughput: 42.5 tokens/sec)**  
Due to its ultra-compact 360M parameter size, `smollm-360m` achieved the fastest generation time and lowest latency (1.266s), nearly 3x faster than `tinyllama` (3.435s). `qwen-0.5b` was close behind at 1.770s (31.8 tokens/sec).

#### 6. Which model requires fewer computational resources?
**Winner: `smollm-360m` (RAM Footprint: 749.2 MB)**  
`smollm-360m` occupied only ~750 MB of RAM, making it ideal for micro-VMs. `qwen-0.5b` required ~1098 MB, while `tinyllama` consumed ~2215 MB.

#### 7. Is the most accurate model also the most efficient? (Quality-Latency-Resource Trade-off)
**No single model dominates all metrics, but `qwen-0.5b` represents the optimal Pareto frontier:**
- `smollm-360m` offers the **highest throughput and lowest memory footprint**, but sacrifices accuracy (-22.1%) and increases hallucinations (+12.8%).
- `tinyllama` consumes the **most memory (~2.2 GB) and has the highest latency (~3.4s)** without surpassing `qwen-0.5b` in accuracy.
- **`qwen-0.5b` is the overall champion**: It provides **top-tier accuracy (0.823)** and **lowest hallucination (0.076)** while requiring only **~1.1 GB RAM** and **1.77s latency**, comfortably fitting in a 5GB RAM VM budget.


In [ ]:
# Visualizing Quality vs. Latency vs. Resource Trade-offs
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

models = list(summary_data.keys())
accuracies = [summary_data[m]["mean_accuracy"] for m in models]
latencies = [summary_data[m]["mean_total_latency_s"] for m in models]
rams = [summary_data[m]["mean_ram_mb"] for m in models]
hallucinations = [summary_data[m]["mean_hallucination_rate"] for m in models]
colors = ["#2b5c8f", "#d95f02", "#7570b3"]

# 1. Accuracy
axes[0, 0].bar(models, accuracies, color=colors, width=0.5)
axes[0, 0].set_title("Accuracy / Correctness (Higher is Better)", fontsize=12, fontweight="bold")
axes[0, 0].set_ylabel("Score (0.0 - 1.0)")
axes[0, 0].set_ylim(0, 1.0)
for i, v in enumerate(accuracies):
    axes[0, 0].text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")

# 2. Total Latency
axes[0, 1].bar(models, latencies, color=colors, width=0.5)
axes[0, 1].set_title("Total Response Latency (Lower is Better)", fontsize=12, fontweight="bold")
axes[0, 1].set_ylabel("Seconds")
for i, v in enumerate(latencies):
    axes[0, 1].text(i, v + 0.08, f"{v:.2f}s", ha="center", fontweight="bold")

# 3. RAM Consumption
axes[1, 0].bar(models, rams, color=colors, width=0.5)
axes[1, 0].set_title("Memory Consumption (Lower is Better)", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("RAM (MB)")
axes[1, 0].axhline(5120, color="red", linestyle="--", label="5 GB VM Limit")
axes[1, 0].legend()
for i, v in enumerate(rams):
    axes[1, 0].text(i, v + 50, f"{v:.1f} MB", ha="center", fontweight="bold")

# 4. Hallucination Rate
axes[1, 1].bar(models, hallucinations, color=colors, width=0.5)
axes[1, 1].set_title("Hallucination Rate (Lower is Better)", fontsize=12, fontweight="bold")
axes[1, 1].set_ylabel("Rate (0.0 - 1.0)")
axes[1, 1].set_ylim(0, 0.35)
for i, v in enumerate(hallucinations):
    axes[1, 1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()


---
## SPECIAL FOCUS: RAG vs. Pure LLM Comparative Analysis

A central theme of evaluating RAG systems is determining the exact quantitative and qualitative advantage of **Retrieval-Augmented Generation (RAG)** over **Pure LLM Generation without retrieval (Parametric Memory only)**.

### Quantitative Comparison Summary:
- **Factual Accuracy**: RAG achieves **82.3%** vs. Pure LLM **43.9%** (**+38.4% improvement**).
- **Hallucination Rate**: RAG reduces hallucinations from **43.8%** down to **7.6%** (**-36.2% reduction**).
- **Private Repository Knowledge**: Pure LLM scores **4.0%** on unreleased repository code questions (guessing generic patterns), while RAG scores **84.0%** by retrieving ground-truth architecture specifications.
- **Unanswerable Query Handling**: On out-of-context or adversarial questions (Q23, Q24), RAG achieves **100% refusal compliance** (*"I don't have enough information"*), whereas Pure LLM answers from pre-training memory or fabricates non-existent algorithms.
- **Latency Overhead**: RAG introduces a mere **~50 ms retrieval overhead** ($t_{\text{retrieval}} \approx 0.052\text{s}$), making the massive quality gains well worth the negligible latency cost.


In [ ]:
# Load and display RAG vs Pure LLM comparison data
with open("week4_rag_vs_llm.json", "r", encoding="utf-8") as f:
    rag_vs_llm_data = json.load(f)

sum_rag = rag_vs_llm_data["summary"]["rag"]
sum_pure = rag_vs_llm_data["summary"]["pure_llm"]

df_rag_vs_llm = pd.DataFrame({
    "Evaluation Metric": [
        "Factual Accuracy (ROUGE & Token-F1)",
        "Hallucination Rate (Fabricated Facts)",
        "Mean Total Latency (s)",
        "Retrieval Step Overhead",
        "Source Citation Capability",
        "Private Codebase Understanding",
        "Unanswerable Query Refusal Rate"
    ],
    "RAG (Retrieved Context)": [
        f"{sum_rag['mean_accuracy']*100:.1f}%",
        f"{sum_rag['mean_hallucination_rate']*100:.1f}%",
        f"{sum_rag['mean_latency_s']:.3f}s",
        f"+{sum_rag['retrieval_overhead_s']*1000:.0f} ms",
        sum_rag["citation_capability"],
        sum_rag["private_codebase_understanding"],
        f"{sum_rag['unanswerable_refusal_rate']*100:.0f}%"
    ],
    "Pure LLM (Zero-Shot / Parametric)": [
        f"{sum_pure['mean_accuracy']*100:.1f}%",
        f"{sum_pure['mean_hallucination_rate']*100:.1f}%",
        f"{sum_pure['mean_latency_s']:.3f}s",
        "0 ms (No retrieval)",
        sum_pure["citation_capability"],
        sum_pure["private_codebase_understanding"],
        f"{sum_pure['unanswerable_refusal_rate']*100:.0f}%"
    ]
})

print("=== RAG VS PURE LLM ARCHITECTURAL COMPARISON ===")
df_rag_vs_llm


---
## EXERCISE 5 – Analyse Your Existing RAG Pipeline


RAG is **not a magic checkbox**. The generation quality of an LLM depends critically on the quality of retrieved context.
We recorded and diagnosed exact traces of:
$$\text{QUESTION} \longrightarrow \text{RETRIEVED CONTEXT} \longrightarrow \text{LLM RESPONSE}$$

### Key RAG Diagnostic Patterns Identified:
1. **Relevant Information Retrieved $\rightarrow$ Accurate LLM Response**:
   - *Example:* Q01 (Transformer self-attention). Dense search retrieves the exact section from `transformer_paper.txt`. The LLM correctly explains multi-head attention and parallelization.
2. **Irrelevant / Distractor Information Retrieved $\rightarrow$ Distracted Answer**:
   - *Example:* Q21 (Dynamic batch sizing). If the vector search returns generic chunking text rather than memory management snippets, the LLM gives vague answers instead of mentioning `psutil` or GPU memory checks.
3. **Important Information Missed $\rightarrow$ Partial or Incomplete Answer**:
   - *Example:* When chunk boundaries split an equation or multi-sentence explanation across two chunks, the retriever may only return the first half, causing the LLM to omit critical parameters.
4. **Context Absent $\rightarrow$ LLM Refusal vs. Hallucination**:
   - *Example:* Q23 (*"What is the capital of France..."*). The retriever returns research papers. A well-behaved RAG model (`qwen-0.5b`) refuses to answer based on the prompt instructions. An unaligned model (`smollm-360m`) ignores context absence and answers from pre-trained weights.


In [ ]:
# Inspect RAG Pipeline Traces
with open("week4_rag_traces.json", "r", encoding="utf-8") as f:
    traces = json.load(f)

df_traces = pd.DataFrame(traces)

print("=== DISTRIBUTION OF RAG PIPELINE DIAGNOSES ===")
print(df_traces["diagnosis"].value_counts().to_string())

# Display sample trace: Accurate context -> Accurate answer
print("\n--- Sample Trace 1: Relevant Context -> Accurate Response ---")
t1 = [t for t in traces if t["question_id"] == "Q01" and t["model"] == "qwen-0.5b"][0]
print(f"Question: {t1['question']}")
print(f"Retrieved Sources: {t1['retrieved_sources']}")
print(f"Retrieved Context Preview:\n{t1['retrieved_context'][:200]}...")
print(f"LLM Response:\n{t1['llm_response']}")
print(f"Diagnosis: {t1['diagnosis']}")

# Display sample trace: Unanswerable query -> Refusal vs Hallucination
print("\n--- Sample Trace 2: Missing Context -> Hallucination vs Refusal ---")
t2_qwen = [t for t in traces if t["question_id"] == "Q23" and t["model"] == "qwen-0.5b"][0]
t2_smol = [t for t in traces if t["question_id"] == "Q23" and t["model"] == "smollm-360m"][0]
print(f"Question: {t2_qwen['question']}")
print(f"[Qwen-0.5B Response]: {t2_qwen['llm_response']} (Diagnosis: {t2_qwen['diagnosis']})")
print(f"[SmolLM-360M Response]: {t2_smol['llm_response']} (Diagnosis: {t2_smol['diagnosis']})")


---
## EXERCISE 6 – Repository / Codebase Understanding

In Week 4, we evaluate whether our existing RAG pipeline can answer questions that require understanding **multiple files, modules, and cross-component interactions** across our repository (`rag_logic.py`, `app.py`, `Dockerfile`).

### Multi-File Repository Questions Tested:
1. **Architecture & File Interaction**:
   - *"Which files and classes are involved in document upload and vector index creation?"*
   - **Ground Truth**: `app.py` receives files via `st.file_uploader`, saves to temporary directory, calls `DocumentProcessor` and `TextChunker` in `rag_logic.py`, generates embeddings via `EmbeddingManager`, and writes to `ChromaVectorStore` and `FAISSVectorStore`.
2. **Dynamic Control Flow**:
   - *"What happens after a user submits a question in app.py?"*
   - **Ground Truth**: `app.py` invokes `RAGPipeline.answer`, which coordinates `SemanticSearcher`, queries `ChromaVectorStore`, builds the context string, and requests `LLMManager.generate`, which signals `ModelCoordinator` to evict any other model before generating text.
3. **Impact Analysis / Refactoring**:
   - *"Which components are affected if the EmbeddingManager interface is modified?"*
   - **Ground Truth**: Affects `SemanticSearcher` (query embedding), `app.py` (index building), and `ModelCoordinator` (manager registration).

### Limitations of Naive Chunk-Based RAG on Codebases:
1. **Loss of Call-Graph & Hierarchical Structure**:
   Standard RAG treats code files as flat text chunks (e.g. 500 characters). A function definition in `rag_logic.py` and its invocation in `app.py` exist in separate chunks with no semantic link.
2. **Boundary Splitting of AST Nodes**:
   Splitting code purely on character counts often severs function signatures from docstrings, imports from usage, or classes from methods.
3. **Inability to Trace Transitive Symbol Dependencies**:
   If module A calls module B which instantiates module C, naive text search retrieves only the surface keyword match without understanding variable types, inheritance, or call chains.

### Motivation for Sourcegraph / Semantic Code Graphs (Week 5 Preview):
To achieve true repository-level code understanding, we must transition from naive text chunking to **graph-based semantic code intelligence**:
- **AST-Aware Parsing**: Chunking code along semantic AST boundaries (whole functions, whole classes).
- **Precise Symbol Navigation**: Using SCIP/LSIF protocols (Find Definitions, Find References, Call Hierarchies).
- **Multi-Repository Code Search**: Sourcegraph semantic code search allows models to navigate dependencies across thousands of files simultaneously.


In [ ]:
# Evaluate Repository-Level Understanding Questions
repo_q_ids = ["Q11", "Q12", "Q13"]
repo_eval = df_results[df_results["question_id"].isin(repo_q_ids)]

print("=== REPOSITORY-LEVEL CODE UNDERSTANDING BENCHMARK RESULTS ===")
print(repo_eval[["model", "question_id", "accuracy", "total_latency_s", "hallucination_rate"]].to_string(index=False))

print("\nMean Accuracy on Repository Understanding Questions by Model:")
print(repo_eval.groupby("model")["accuracy"].mean())


---
## Conclusion & Key Takeaways

1. **Systematic Model Selection Matters**:
   - Moving from `Phi-2` (2.7B) to `Qwen2.5-0.5B` eliminated VM OOM crashes while maintaining high factual accuracy ($0.823$) and lowest hallucination ($0.076$).
   - `ModelCoordinator` successfully enforces single-model memory residence, keeping total peak RAM to **~1.1 GB** (well below the 5 GB VM threshold).
2. **RAG Diagnosis Confirms Context Sensitivity**:
   - RAG response quality is bounded by retrieval precision. Irrelevant or missing context leads to hallucinations unless the model is trained to recognize knowledge boundaries.
3. **Repository Understanding Requires Graph-Aware Tools**:
   - Naive text chunking is insufficient for cross-file architecture and call-graph reasoning, establishing the clear need for semantic code navigation tools like Sourcegraph in Week 5.
